# AES-128: merhaba
Eğitim örneği: AES-128 + PKCS#7, blokları bağımsız işleyen ECB düzeni. Sabit anahtar ve ECB yalnızca adımları tekrar edebilmek içindir. Gerçek uygulamada doğrulanmış bir kütüphane ve AES-GCM gibi kimlik doğrulamalı bir kip kullanılır. Hex şifreleme değil, baytların yazım biçimidir.
Hücreleri yukarıdan aşağıya çalıştır. İlk kurulum hücresi dış paket gerektirmeyen eğitim motorunu tanımlar.

In [ ]:
def gf_mul(a, b):
    """GF(2^8) çarpımı; AES indirgeme polinomu 0x11b."""
    result = 0
    for _ in range(8):
        if b & 1:
            result ^= a
        a = ((a << 1) ^ (0x11b if a & 0x80 else 0)) & 0xff
        b >>= 1
    return result


def make_sbox():
    def inverse(x):
        if x == 0:
            return 0
        y = 1
        for _ in range(254):
            y = gf_mul(y, x)
        return y

    def rotate(x, n):
        return ((x << n) | (x >> (8 - n))) & 255

    table = []
    for x in range(256):
        y = inverse(x)
        table.append(y ^ rotate(y, 1) ^ rotate(y, 2) ^ rotate(y, 3) ^ rotate(y, 4) ^ 0x63)
    return table


SBOX = make_sbox()
INV_SBOX = [SBOX.index(x) for x in range(256)]


def sub_bytes(state, inverse=False):
    table = INV_SBOX if inverse else SBOX
    return [table[x] for x in state]


def shift_rows(state, inverse=False):
    # Sütun öncelikli: state[4 * sütun + satır].
    direction = -1 if inverse else 1
    return [state[4 * ((c + direction * r) % 4) + r]
            for c in range(4) for r in range(4)]


def mix_columns(state, inverse=False):
    matrix = ([[14, 11, 13, 9], [9, 14, 11, 13],
               [13, 9, 14, 11], [11, 13, 9, 14]] if inverse else
              [[2, 3, 1, 1], [1, 2, 3, 1], [1, 1, 2, 3], [3, 1, 1, 2]])
    result = []
    for c in range(4):
        column = state[4*c:4*c+4]
        for row in matrix:
            value = 0
            for coefficient, byte in zip(row, column):
                value ^= gf_mul(coefficient, byte)
            result.append(value)
    return result


def add_round_key(state, key):
    return [a ^ b for a, b in zip(state, key)]


def expand_key(key):
    if len(key) != 16:
        raise ValueError("AES-128 anahtarı tam 16 bayt olmalı.")
    words = [list(key[i:i+4]) for i in range(0, 16, 4)]
    rcon = 1
    for i in range(4, 44):
        temp = words[i-1].copy()
        if i % 4 == 0:
            temp = temp[1:] + temp[:1]  # RotWord
            temp = [SBOX[x] for x in temp]  # SubWord
            temp[0] ^= rcon
            rcon = gf_mul(rcon, 2)
        words.append([a ^ b for a, b in zip(words[i-4], temp)])
    return [sum(words[i:i+4], []) for i in range(0, 44, 4)]


def crypt_block(block, key, decrypt=False):
    if len(block) != 16:
        raise ValueError("AES bloğu tam 16 bayt olmalı.")
    keys = expand_key(key)
    state = list(block)
    trace = []

    def record(name, round_no, before, after, round_key=None):
        trace.append(dict(name=name, round=round_no, before=list(before),
                          after=list(after), key=round_key))

    def apply(name, round_no, operation, round_key=None):
        nonlocal state
        before = state.copy()
        state = operation(state)
        record(name, round_no, before, state, round_key)

    record("Şifreli blok" if decrypt else "Dolgulu blok", 10 if decrypt else 0, state, state)
    if not decrypt:
        apply("AddRoundKey", 0, lambda s: add_round_key(s, keys[0]), keys[0])
        for r in range(1, 11):
            apply("SubBytes", r, sub_bytes)
            apply("ShiftRows", r, shift_rows)
            if r != 10:
                apply("MixColumns", r, mix_columns)
            apply("AddRoundKey", r, lambda s: add_round_key(s, keys[r]), keys[r])
    else:
        apply("AddRoundKey", 10, lambda s: add_round_key(s, keys[10]), keys[10])
        for r in range(9, -1, -1):
            apply("InvShiftRows", r, lambda s: shift_rows(s, True))
            apply("InvSubBytes", r, lambda s: sub_bytes(s, True))
            apply("AddRoundKey", r, lambda s: add_round_key(s, keys[r]), keys[r])
            if r != 0:
                apply("InvMixColumns", r, lambda s: mix_columns(s, True))
    return bytes(state), trace


def pad(data):
    n = 16 - len(data) % 16
    return data + bytes([n]) * n


def unpad(data):
    if not data or len(data) % 16:
        raise ValueError("Geçersiz dolgulu veri.")
    n = data[-1]
    if not 1 <= n <= 16 or data[-n:] != bytes([n]) * n:
        raise ValueError("Geçersiz PKCS#7 dolgusu.")
    return data[:-n]


def aes_encrypt(text, key):
    data = pad(text.encode("utf-8"))
    return b"".join(crypt_block(data[i:i+16], key)[0] for i in range(0, len(data), 16))


def aes_decrypt(ciphertext, key):
    if not ciphertext or len(ciphertext) % 16:
        raise ValueError("Şifreli veri 16 baytın pozitif katı olmalı.")
    data = b"".join(crypt_block(ciphertext[i:i+16], key, True)[0]
                    for i in range(0, len(ciphertext), 16))
    return unpad(data).decode("utf-8")


def show_state(state):
    for r in range(4):
        print(" ".join(f"{state[4*c+r]:02x}" for c in range(4)))


## 1. Doğrudan şifrele ve çöz
Anahtar tam 16 ASCII bayt: 0123456789abcdef. Önce sonucu görelim. Alttaki hücrelerde aynı işlemin içini açacağız.

In [ ]:
text = "merhaba"
key = b"0123456789abcdef"
ciphertext = aes_encrypt(text, key)
decrypted = aes_decrypt(ciphertext, key)
print("Açık metin :", text)
print("Anahtar    :", key.decode())
print("Şifre (hex):", ciphertext.hex())
print("Çözülmüş   :", decrypted)
assert decrypted == text

## 2. Metinden baytlara ve dolguya
AES harfleri kaydırmaz; 16 baytlık blokları dönüştürür. UTF-8 ile merhaba 7 bayttır. PKCS#7, kalan 9 yere 09 koyar. Hex gösterimde iki basamak bir bayttır. Dolgu AES çekirdeğinden ayrı bir hazırlıktır.

In [ ]:
raw = text.encode("utf-8")
padded = pad(raw)
print("UTF-8:", raw.hex(" "))
print("Bayt sayısı:", len(raw))
print("Dolgu sayısı:", 16 - len(raw) % 16)
print("Dolgulu veri:", padded.hex(" "))
print("İlk blok, sütun öncelikli 4×4 state:")
show_state(padded[:16])

## 3. Anahtarı 11 tur anahtarına genişlet
AES-128: başlangıç anahtarı K0 ve 10 tur için K1–K10. Her kelime 4 bayttır. Her dördüncü kelimede RotWord → SubWord → Rcon XOR uygulanır; sonra dört kelime önceki değerle XOR yapılır.

In [ ]:
keys = expand_key(key)
for r, round_key in enumerate(keys):
    print(f"K{r:02}:", bytes(round_key).hex(" "))
word = list(key[12:16])
rotated = word[1:] + word[:1]
substituted = [SBOX[x] for x in rotated]
with_rcon = substituted.copy()
with_rcon[0] ^= 1
print("Son kelime:", bytes(word).hex(" "))
print("RotWord   :", bytes(rotated).hex(" "))
print("SubWord   :", bytes(substituted).hex(" "))
print("Rcon XOR  :", bytes(with_rcon).hex(" "))
print("Yeni kelime:", bytes(a ^ b for a,b in zip(key[:4], with_rcon)).hex(" "))

## 4. Başlangıç: AddRoundKey
State ile K0 bayt bayt XOR yapılır. XOR aynı anahtarla tekrar uygulanınca başlangıç değeri geri gelir.

In [ ]:
state = list(padded[:16])
print(f"İlk bayt: {state[0]:02x} XOR {keys[0][0]:02x} = {state[0] ^ keys[0][0]:02x}")
print(f"Bitlerle : {state[0]:08b} XOR {keys[0][0]:08b} = {state[0] ^ keys[0][0]:08b}")
state = add_round_key(state, keys[0])
show_state(state)

## 5. SubBytes: her baytı S-box ile değiştir
S-box, 256 girişli doğrusal olmayan bir tablodur. Örneğin 5d için satır 5, sütun d kullanılır. Kodda SBOX[0x5d] okunur. Tablonun üretimi aşağıdaki kaynak kodda yer alır.

In [ ]:
print("Önce (SubBytes yok):")
show_state(state)

before = state.copy()

# Örnek: ilk baytı elle S-box'tan oku
b = before[0]
row = b >> 4          # üst 4 bit = satır
col = b & 0x0f        # alt 4 bit = sütun
print()
print(f"Örnek bayt: {b:02x}")
print(f"  satır = {row:x}, sütun = {col:x}")
print(f"  SBOX[{b:02x}] = {SBOX[b]:02x}")

print()
print("Sıra\tÖnce\tSonra")
for i, x in enumerate(before[:8]):
    print(f"{i}\t{x:02x}\t{SBOX[x]:02x}")

state = sub_bytes(state)

print()
print("Sonra (her bayt S-box ile değişti):")
show_state(state)


## 6. ShiftRows: satırları sola döndür
0. satır değişmez. 1., 2., 3. satırlar sırasıyla 1, 2, 3 hücre sola döner. Baytlar değişmez, konumları değişir.

In [ ]:
print("Önce:")
show_state(state)
state = shift_rows(state)
print("Sonra:")
show_state(state)

## 7. MixColumns: her sütunu karıştır
Her sütun sabit bir matrisle GF(2⁸) içinde çarpılır. Toplama XOR’dur. İlk çıktı = (02·a) XOR (03·b) XOR c XOR d. Diğer satırların katsayıları: 01 02 03 01; 01 01 02 03; 03 01 01 02. Çarpım normal tamsayı çarpımı değildir; indirgeme polinomu 0x11b’dir.

In [ ]:
print("Önce (MixColumns yok):")
show_state(state)

# İlk sütun: a,b,c,d (yukarıdan aşağıya)
a, b, c, d = state[0], state[1], state[2], state[3]
print()
print(f"İlk sütun: {a:02x}  {b:02x}  {c:02x}  {d:02x}")
print()
print("Sabit matris (her sütun için):")
print("  [02 03 01 01]")
print("  [01 02 03 01]")
print("  [01 01 02 03]")
print("  [03 01 01 02]")
print()
print("Toplama = XOR, çarpma = gf_mul (AES alanı)")
print()

y0 = gf_mul(2, a) ^ gf_mul(3, b) ^ c ^ d
y1 = a ^ gf_mul(2, b) ^ gf_mul(3, c) ^ d
y2 = a ^ b ^ gf_mul(2, c) ^ gf_mul(3, d)
y3 = gf_mul(3, a) ^ b ^ c ^ gf_mul(2, d)

print("İlk sütunun yeni değerleri:")
print(f"  y0 = 02·{a:02x} XOR 03·{b:02x} XOR {c:02x} XOR {d:02x} = {y0:02x}")
print(f"  y1 = {a:02x} XOR 02·{b:02x} XOR 03·{c:02x} XOR {d:02x} = {y1:02x}")
print(f"  y2 = {a:02x} XOR {b:02x} XOR 02·{c:02x} XOR 03·{d:02x} = {y2:02x}")
print(f"  y3 = 03·{a:02x} XOR {b:02x} XOR {c:02x} XOR 02·{d:02x} = {y3:02x}")

state = mix_columns(state)

print()
print("Sonra (4 sütunun hepsi aynı kuralla karıştı):")
show_state(state)
print(f"Kontrol: ilk sütun = {state[0]:02x} {state[1]:02x} {state[2]:02x} {state[3]:02x}")


## 8. Tur anahtarını ekle; 10 turu tamamla
1–9. turlar: SubBytes → ShiftRows → MixColumns → AddRoundKey. 10. turda MixColumns yoktur. Aşağıdaki iz, ilk blok için bütün ara değerleri yeniden hesaplar.

In [ ]:
state = add_round_key(state, keys[1])
print("1. tur sonu:")
show_state(state)
block_cipher, encryption_trace = crypt_block(padded[:16], key)
for step in encryption_trace:
    print(f"Tur {step['round']:2} {step['name']:15} {bytes(step['after']).hex(' ')}")
print("İlk blok şifre:", block_cipher.hex())
assert block_cipher == ciphertext[:16]

## 9. Çözme: ters işlemler ve dolguyu kaldırma
Önce K10 ile XOR. Ardından InvShiftRows → InvSubBytes → AddRoundKey → InvMixColumns uygulanır. Son çözme turunda InvMixColumns yoktur. Tur anahtarları ters sırada kullanılır. Son olarak PKCS#7 kaldırılır ve UTF-8 çözülür.

In [ ]:
recovered_block, decryption_trace = crypt_block(block_cipher, key, decrypt=True)
for step in decryption_trace:
    print(f"Tur {step['round']:2} {step['name']:15} {bytes(step['after']).hex(' ')}")
print("Geri gelen ilk blok:", recovered_block.hex(" "))
print("Dolgu kaldırılıp UTF-8 çözülünce:", aes_decrypt(ciphertext, key))
assert recovered_block == padded[:16]